# Predictive Coding Network — Convolutional MNIST Demo

Demonstrates a convolutional PCN architecture on MNIST, using Conv layers,
MaxPool layers, and a final Linear classification head with cross-entropy energy.

## Imports & JAX Setup

In [ ]:
import time
import jax
import optax

from fabricpc.nodes import ConvNode, MaxPool, Linear, IdentityNode
from fabricpc.core.topology import Edge
from fabricpc.graph_assembly import TaskMap, graph
from fabricpc.graph_initialization import initialize_params
from fabricpc.core.activations import LeakyReLUActivation, SoftmaxActivation
from fabricpc.core.energy import CrossEntropyEnergy, GaussianEnergy
from fabricpc.core.inference import InferenceSGD
from fabricpc.core.initializers import KaimingInitializer, XavierInitializer
from fabricpc.training import train_pcn, evaluate_pcn
from fabricpc.utils.data.dataloader import MnistLoader

jax.config.update("jax_default_prng_impl", "threefry2x32")

## Define the Network

Convolutional architecture with MaxPool layers.

In [ ]:
pixels = IdentityNode(shape=(28, 28, 1), name="pixels")

conv1 = ConvNode(
    shape=(28, 28, 32),
    kernel_size=(3, 3),
    stride=(1, 1),
    padding="SAME",
    activation=LeakyReLUActivation(alpha=0.01),
    energy=GaussianEnergy(),
    weight_init=KaimingInitializer(nonlinearity="leaky_relu", a=0.01),
    name="conv1",
)

pool1 = MaxPool(
    shape=(14, 14, 32),
    window_shape=(2, 2),
    stride=(2, 2),
    padding="VALID",
    name="pool1",
)

conv2 = ConvNode(
    shape=(14, 14, 64),
    kernel_size=(3, 3),
    stride=(1, 1),
    padding="SAME",
    activation=LeakyReLUActivation(alpha=0.01),
    energy=GaussianEnergy(),
    weight_init=KaimingInitializer(nonlinearity="leaky_relu", a=0.01),
    name="conv2",
)

pool2 = MaxPool(
    shape=(7, 7, 64),
    window_shape=(2, 2),
    stride=(2, 2),
    padding="VALID",
    name="pool2",
)

conv3 = ConvNode(
    shape=(7, 7, 128),
    kernel_size=(3, 3),
    stride=(1, 1),
    padding="SAME",
    activation=LeakyReLUActivation(alpha=0.01),
    energy=GaussianEnergy(),
    weight_init=KaimingInitializer(nonlinearity="leaky_relu", a=0.01),
    name="conv3",
)

output = Linear(
    shape=(10,),
    activation=SoftmaxActivation(),
    energy=CrossEntropyEnergy(),
    flatten_input=True,
    weight_init=XavierInitializer(),
    name="class",
)

structure = graph(
    nodes=[pixels, conv1, pool1, conv2, pool2, conv3, output],
    edges=[
        Edge(source=pixels, target=conv1.slot("in")),
        Edge(source=conv1, target=pool1.slot("in")),
        Edge(source=pool1, target=conv2.slot("in")),
        Edge(source=conv2, target=pool2.slot("in")),
        Edge(source=pool2, target=conv3.slot("in")),
        Edge(source=conv3, target=output.slot("in")),
    ],
    task_map=TaskMap(x=pixels, y=output),
    inference=InferenceSGD(eta_infer=0.01, infer_steps=100),
)

## Hyperparameters

In [ ]:
train_config = {"num_epochs": 20}
batch_size = 200
optimizer = optax.adamw(0.001, weight_decay=0.01)

## Initialize Parameters & Data Loaders

In [ ]:
master_rng_key = jax.random.PRNGKey(42)
graph_key, train_key, eval_key = jax.random.split(master_rng_key, 3)

params = initialize_params(structure, graph_key)

train_loader = MnistLoader(
    "train", batch_size=batch_size, tensor_format="nhwc", shuffle=True, seed=42
)
test_loader = MnistLoader(
    "test", batch_size=batch_size, tensor_format="nhwc", shuffle=False
)

print(
    f"\n{len(structure.nodes)} nodes, {len(structure.edges)} edges, "
    f"{sum(p.size for p in jax.tree_util.tree_leaves(params)):,} parameters"
)

## Train

In [ ]:
print("\nTraining (JIT compilation on first batch)...")
start_time = time.time()
trained_params, energy_history, _ = train_pcn(
    params=params,
    structure=structure,
    train_loader=train_loader,
    optimizer=optimizer,
    config=train_config,
    rng_key=train_key,
    verbose=True,
)
elapsed = time.time() - start_time
print(f"Avg training time: {elapsed / train_config['num_epochs']:.2f}s per epoch")

## Evaluate

In [ ]:
print("\nEvaluating...")
metrics = evaluate_pcn(
    trained_params, structure, test_loader, train_config, eval_key
)
print(f"Test Accuracy: {metrics['accuracy'] * 100:.2f}%")